# 06 — Native Station Value Extraction (FIXED)

Creates a tidy modelling table with one row per station-year-month.

**Key correction:** precipitation values are sampled from the **raw/native**
raster of the matching year/month. No training precipitation is first resampled
to an NDVI grid.

- Outside-raster station → `NaN` + QC flag (never edge-clamped).
- Raster NoData → `NaN` (never nearest-filled).
- Dynamic variables are matched to the same year/month.
- Static variables are sampled once per station.

In [ ]:
from pathlib import Path
import warnings

def find_project_root(start=None):
    current = Path(start or Path.cwd()).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "data").exists():
            return candidate
    raise FileNotFoundError(
        "Project root not found. Run this notebook from inside the repository."
    )

PROJECT_ROOT = find_project_root()
DATA_DIR = PROJECT_ROOT / "data"
RAW_DIR = DATA_DIR / "raw"
INTERIM_DIR = DATA_DIR / "interim"
PROCESSED_DIR = DATA_DIR / "processed"
OUTPUT_DIR = PROJECT_ROOT / "outputs"
MODEL_DIR = PROJECT_ROOT / "models"

for d in [INTERIM_DIR, PROCESSED_DIR, OUTPUT_DIR, MODEL_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT =", PROJECT_ROOT)

In [ ]:
import re
import numpy as np
import pandas as pd
import rasterio
from pyproj import Transformer, CRS

GAUGE_PATH = PROCESSED_DIR / "gauge_monthly_clean.csv"
if not GAUGE_PATH.exists():
    raise FileNotFoundError("Run 02_Gauge_Preprocessing_FIXED.ipynb first.")
gauge = pd.read_csv(GAUGE_PATH)

PRECIP_PRODUCTS = ["CCS","PDIR","GSMaP_MVK","CDR","CHIRPS","IMERG","GSMaP_Gauge_v7","ERA5"]
DYNAMIC_LAND = ["NDVI","LST_Day"]

def parse_ym(name):
    stem = Path(name).stem
    for pat in [r"(?<!\d)(20\d{2})[_-](0?[1-9]|1[0-2])(?!\d)",
                r"(?<!\d)(20\d{2})(0[1-9]|1[0-2])(?!\d)"]:
        m = re.search(pat, stem)
        if m:
            return int(m.group(1)), int(m.group(2))
    return None

def list_rasters(folder):
    return sorted([*folder.rglob("*.tif"), *folder.rglob("*.tiff")])

def monthly_map(folder):
    out = {}
    for p in list_rasters(folder):
        ym = parse_ym(p.name)
        if ym:
            if ym in out:
                raise ValueError(f"Duplicate {folder.name} raster for {ym}")
            out[ym]=p
    return out

def choose_static(folder_name, preferred_names):
    folder = RAW_DIR/"predictors"/folder_name
    files = list_rasters(folder)
    d = {p.name.lower():p for p in files}
    for n in preferred_names:
        if n.lower() in d:
            return d[n.lower()]
    clean = [p for p in files if not any(x in p.stem.lower() for x in ["clip","tmp","temp","aligned","resampl"])]
    if len(clean)==1: return clean[0]
    if len(files)==1: return files[0]
    raise ValueError(f"Ambiguous {folder_name} raster. Choose one manually: {files}")

DEM_PATH = choose_static("DEM", ["Khulna_SRTM_DEM.tif","DEM.tif"])
DFS_PATH = choose_static("Distance_Sea", ["Distance_Sea.tif","distance_to_sea.tif"])

precip_maps = {p:monthly_map(RAW_DIR/"precipitation"/p) for p in PRECIP_PRODUCTS}
land_maps = {p:monthly_map(RAW_DIR/"predictors"/p) for p in DYNAMIC_LAND}

In [ ]:
def resolved_crs(src, path):
    if src.crs is not None:
        return src.crs
    b = src.bounds
    if (-180 <= b.left <= 180 and -180 <= b.right <= 180 and
        -90 <= b.bottom <= 90 and -90 <= b.top <= 90):
        warnings.warn(f"{path.name}: missing CRS, geographic-looking bounds -> assuming EPSG:4326.")
        return CRS.from_epsg(4326)
    raise ValueError(f"{path}: missing CRS and cannot safely infer WGS84.")

def sample_native(path, lon, lat):
    with rasterio.open(path) as src:
        src_crs = resolved_crs(src, path)
        if str(src_crs).upper() in ["EPSG:4326", "OGC:CRS84"]:
            x,y = float(lon),float(lat)
        else:
            tr = Transformer.from_crs("EPSG:4326", src_crs, always_xy=True)
            x,y = tr.transform(float(lon),float(lat))

        b = src.bounds
        if not (b.left <= x <= b.right and b.bottom <= y <= b.top):
            return np.nan, "outside"

        row,col = src.index(x,y)
        if row < 0 or row >= src.height or col < 0 or col >= src.width:
            return np.nan, "outside"

        a = src.read(1, window=((row,row+1),(col,col+1)), masked=True)
        if a.mask.all():
            return np.nan, "nodata"

        val = float(a[0,0])
        if not np.isfinite(val):
            return np.nan, "nodata"
        if src.nodata is not None and np.isclose(val, src.nodata):
            return np.nan, "nodata"
        return val, "ok"

In [ ]:
# Static values by station.
station_static = {}
static_status = []
for station, s in gauge.groupby("station_id", sort=False):
    lon = float(s.longitude.median())
    lat = float(s.latitude.median())
    dem, st_dem = sample_native(DEM_PATH, lon, lat)
    dfs, st_dfs = sample_native(DFS_PATH, lon, lat)
    station_static[station] = {"DEM":dem, "Distance_Sea":dfs}
    static_status.append({"station_id":station,"DEM_status":st_dem,"Distance_Sea_status":st_dfs})

display(pd.DataFrame(static_status))

In [ ]:
records = []
qc_records = []

for _, r in gauge.iterrows():
    station = r["station_id"]
    y,m = int(r["year"]), int(r["month"])
    lon,lat = float(r["longitude"]),float(r["latitude"])
    rec = {
        "station_id":station, "year":y, "month":m,
        "date":f"{y}-{m:02d}-01",
        "latitude":lat, "longitude":lon,
        "rainfall_mm":float(r["rainfall_mm"]),
    }
    rec.update(station_static[station])

    for product in PRECIP_PRODUCTS:
        path = precip_maps[product].get((y,m))
        if path is None:
            val,status=np.nan,"missing_file"
        else:
            val,status=sample_native(path,lon,lat)
        # Precipitation cannot be negative; negative values are treated as invalid/no-data.
        if np.isfinite(val) and val < 0:
            val,status=np.nan,"negative_invalid"
        rec[product]=val
        qc_records.append({"station_id":station,"year":y,"month":m,"feature":product,"status":status})

    for pred in DYNAMIC_LAND:
        path = land_maps[pred].get((y,m))
        if path is None:
            val,status=np.nan,"missing_file"
        else:
            val,status=sample_native(path,lon,lat)
        rec[pred]=val
        qc_records.append({"station_id":station,"year":y,"month":m,"feature":pred,"status":status})

    records.append(rec)

samples = pd.DataFrame(records).sort_values(["station_id","year","month"]).reset_index(drop=True)
qc = pd.DataFrame(qc_records)

print("Tidy modelling table:", samples.shape)
display(samples.head())
print("\nMissing values by feature:")
display(samples.isna().sum().to_frame("missing"))
print("\nExtraction status:")
display(qc.groupby(["feature","status"]).size().rename("n").reset_index())

samples.to_csv(PROCESSED_DIR/"station_samples_native_tidy.csv", index=False)
qc.to_csv(PROCESSED_DIR/"station_extraction_qc.csv", index=False)

print("Saved:", PROCESSED_DIR/"station_samples_native_tidy.csv")

In [ ]:
# Stop if a required feature has excessive missingness.
required_features = PRECIP_PRODUCTS + ["DEM","NDVI","LST_Day","Distance_Sea"]
missing_pct = samples[required_features].isna().mean()*100
display(missing_pct.to_frame("missing_pct"))

if (missing_pct > 20).any():
    print("WARNING: >20% missing values in one or more required features.")
    print("Fix source coverage/CRS/NoData rather than filling large gaps with nearest values.")